# Day 1 – Exercise Solutions

This notebook contains solutions for the optional exercises.

**Note:** Run the code cells sequentially. Some cells require a Hugging Face API token or a GPU.

---

## Exercise 1: Prompt diversity with GPT‑2

Generate text for five different prompts at two temperatures (0.2 and 1.0).

**Observation:** Lower temperature produces more repetitive, conservative outputs; higher temperature introduces more creativity and unexpected word choices.

In [ ]:
from transformers import pipeline

# Load GPT-2 pipeline
generator = pipeline('text-generation', model='gpt2')

# Define prompts
prompts = [
    "Once upon a time",
    "The secret to happiness is",
    "If I could travel anywhere",
    "The most important invention of the 21st century is",
    "In a world where robots rule"
]

temperatures = [0.2, 1.0]

for prompt in prompts:
    print(f"\n{'='*60}\nPrompt: {prompt}\n")
    for temp in temperatures:
        output = generator(prompt, max_length=50, temperature=temp, do_sample=True)
        print(f"--- temperature = {temp} ---")
        print(output[0]['generated_text'])
        print()

**Sample observation:**
- At `temperature=0.2`, outputs are safe, common, and sometimes repetitive.
- At `temperature=1.0`, outputs are surprising, occasionally off‑topic, but more creative.

## Exercise 2: Compare image generation models

Generate the same prompt using two different Stable Diffusion models.

**Note:** This will download each model (~2GB each) on first run. Use a GPU runtime for speed.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from IPython.display import display

# Models to compare
models = {
    "Realistic": "dreamlike-art/dreamlike-photoreal-2.0",
    "Artistic": "prompthero/openjourney"
}

prompt = "a robot reading a book in a library, detailed"

for style, model_id in models.items():
    print(f"\nGenerating with {style} model: {model_id}")
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16
    )
    pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
    
    image = pipe(prompt, guidance_scale=7.5).images[0]
    display(image)
    # Uncomment to save: image.save(f"{style}_robot_library.png")

**Comparison notes:**
- Dreamlike Photorealistic produces more lifelike textures and lighting.
- OpenJourney gives a softer, painterly, anime‑influenced style.

Choice depends on the desired aesthetic – photorealistic for realism, openjourney for artistic / illustrative projects.

## Exercise 3: Use the Hugging Face Inference API

Call GPT‑2 and Stable Diffusion via the free Inference API (requires token).

**Setup:** Replace `"YOUR_HF_TOKEN"` with your actual token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
import requests
import base64
from PIL import Image
from io import BytesIO

HF_TOKEN = "YOUR_HF_TOKEN"  # Replace with your token
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

# --- Text generation with GPT-2 ---
text_api_url = "https://api-inference.huggingface.co/models/gpt2"
text_payload = {
    "inputs": "The meaning of life is",
    "parameters": {"max_length": 50, "temperature": 0.8}
}

response = requests.post(text_api_url, headers=headers, json=text_payload)
if response.status_code == 200:
    result = response.json()
    print("GPT-2 output:", result[0]['generated_text'])
else:
    print(f"Error {response.status_code}: {response.text}")

In [ ]:
# --- Image generation with Stable Diffusion 2.1 ---
image_api_url = "https://api-inference.huggingface.co/models/stabilityai/stable-diffusion-2-1"
image_payload = {
    "inputs": "a serene lake at sunset, mountains in background, high quality"
}

response = requests.post(image_api_url, headers=headers, json=image_payload)

if response.status_code == 200:
    # Decode the returned image (base64 or raw bytes)
    image_bytes = response.content
    image = Image.open(BytesIO(image_bytes))
    display(image)
    # image.save("api_generated_sunset.png")
else:
    print(f"Error {response.status_code}: {response.text}")
    print("Note: The model may take ~20 seconds to load on first call.")

**Troubleshooting:**
- If you get a 401 error, check your token.
- If the model is loading, wait a few seconds and retry (the API returns a 503 with an `estimated_time`).

## Exercise 4: Reflection essay (sample)

*“How might generative AI change content creation in the next 5 years?”*

**Positive changes:**
1. **Democratisation of creativity** – People without technical skills will generate high‑quality art, music, and writing. Small businesses can produce professional marketing materials instantly.
2. **Hyper‑personalisation** – Education and entertainment will adapt in real time to each user’s preferences, creating truly personalised learning and media experiences.

**Potential negative change:**
- **Misinformation & deepfakes** – As generated content becomes indistinguishable from reality, trust in digital media may erode. Malicious actors could flood the internet with convincing fake news, requiring robust detection and authentication systems.

Balancing innovation with safeguards will be the key challenge of the next decade.